<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/10_aktivasyon_yonlendirme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aktivasyon Yönlendirme (Activation Steering)

Bir LLM'in davranışını değiştirmenin bilinen yolları **istem mühendisliği** (girdiyi değiştir) ve **ince ayardır** (ağırlıkları değiştir). Aktivasyon yönlendirme üçüncü bir yol sunar: ağırlıklara ve isteme dokunmadan, modelin **ara katman aktivasyonlarına** (residual stream) çıkarım sırasında bir **yön vektörü** ekleyerek davranışını kaydırmak.

| Yöntem | Neyi değiştirir? | Maliyet | Geri alınabilir mi? |
|---|---|---|---|
| İstem mühendisliği | Girdi metni | Sıfır | Evet |
| İnce ayar (LoRA vb.) | Ağırlıklar | Eğitim gerekir | Adaptörü çıkararak |
| **Aktivasyon yönlendirme** | Ara katman aktivasyonları | Birkaç ileri geçiş | Kancayı (hook) kaldırarak, anında |

**Temel varsayım — doğrusal temsil hipotezi:** Modeller "duygu", "resmiyet", "dil", "doğruluk" gibi yüksek seviyeli kavramları aktivasyon uzayında yaklaşık olarak **doğrusal yönler** olarak kodlar. O yönü bulursak, aktivasyonu o yönde iterek kavramı güçlendirebilir veya zayıflatabiliriz.

Bu defterde:
1. Bir modelin ara katman aktivasyonlarını okumak
2. Zıt örnek çiftlerinden **yönlendirme vektörü** çıkarmak (*Contrastive Activation Addition*)
3. Kavramın gerçekten doğrusal olarak ayrılabilir olduğunu bir **doğrusal sonda (linear probe)** ile test etmek
4. En iyi katmanı seçmek
5. PyTorch **forward hook** ile üretim sırasında aktivasyonları yönlendirmek ve katsayı (α) taraması yapmak

> Model: `Qwen/Qwen2.5-0.5B-Instruct` (~1 GB). Colab'da `T4 GPU` ile hızlı çalışır; CPU'da da çalışır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ADI = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForCausalLM.from_pretrained(MODEL_ADI).to(cihaz)
if cihaz == "cuda":
    model = model.half()
model.eval()

katmanlar = model.model.layers
N_KATMAN, D_MODEL = len(katmanlar), model.config.hidden_size
print(f"Cihaz: {cihaz} | katman sayısı: {N_KATMAN} | gizli boyut: {D_MODEL}")

---
## 1) Residual stream: modelin "çalışma belleği"

Bir decoder Transformer'da her token için bir vektör katmandan katmana akar. Her blok bu vektöre bir şey **ekler**:

$$h^{(\ell+1)} = h^{(\ell)} + \text{Attn}^{(\ell)}(h^{(\ell)}) + \text{MLP}^{(\ell)}(\cdot)$$

Bu toplamsal yapı yüzünden residual stream'e dışarıdan bir vektör eklemek, sonraki tüm katmanların "okuyacağı" bir sinyal enjekte etmek demektir. Aktivasyon yönlendirme tam olarak bunu kullanır:

$$h^{(\ell)} \leftarrow h^{(\ell)} + \alpha \cdot \mathbf{v}$$

`output_hidden_states=True` ile her katmanın çıktısını okuyabiliriz:

In [ ]:
@torch.no_grad()
def aktivasyonlar(metinler, birlestirme="ortalama"):
    """Her metin için her katmanın residual stream vektörünü döndürür: (n_metin, n_katman, d_model)"""
    sonuc = []
    for metin in metinler:
        girdi = tokenizer(metin, return_tensors="pt").to(cihaz)
        gizli = model(**girdi, output_hidden_states=True).hidden_states   # (n_katman+1) × (1, T, d)
        katman_ciktilari = torch.stack(gizli[1:], dim=0)[:, 0]          # (n_katman, T, d) — gömme katmanı hariç
        vektor = katman_ciktilari.mean(1) if birlestirme == "ortalama" else katman_ciktilari[:, -1]
        sonuc.append(vektor.float().cpu())
    return torch.stack(sonuc)

ornek = aktivasyonlar(["Bugün hava çok güzel."])
print("Aktivasyon tensörü:", tuple(ornek.shape), "→ (metin, katman, d_model)")
print("Katmanlara göre vektör normu:", ornek[0].norm(dim=-1).numpy().round(0)[::4], "...")

---
## 2) Zıt örnek çiftleri (contrastive pairs)

Yönlendirmek istediğimiz kavram: **olumlu ↔ olumsuz duygu**. Aynı konudan bahseden ama duygusu zıt cümle çiftleri hazırlıyoruz. Çiftlerin konu bakımından eşleşmesi önemli: farkı aldığımızda konu bilgisi birbirini götürür, geriye ağırlıklı olarak **duygu yönü** kalır.

$$\mathbf{v}^{(\ell)} = \frac{1}{N}\sum_{i=1}^{N}\left(h^{(\ell)}(\text{olumlu}_i) - h^{(\ell)}(\text{olumsuz}_i)\right)$$

Bu, *Contrastive Activation Addition* (CAA, Rimsky vd., 2024) ve *ActAdd* (Turner vd., 2023) yöntemlerinin özüdür.

In [ ]:
ciftler = [
    ("Bu film harikaydı, her dakikasından keyif aldım.", "Bu film berbattı, her dakikası işkence gibiydi."),
    ("Yemekler çok lezzetliydi, kesinlikle tekrar geleceğim.", "Yemekler çok kötüydü, bir daha asla gelmem."),
    ("Yeni işimde çok mutluyum, ekibim harika.", "Yeni işimde çok mutsuzum, ekibim berbat."),
    ("Tatil muhteşem geçti, her şey mükemmeldi.", "Tatil felaket geçti, her şey ters gitti."),
    ("Bu telefon beklentilerimi fazlasıyla karşıladı.", "Bu telefon tam bir hayal kırıklığı oldu."),
    ("Konser büyüleyiciydi, sanatçı inanılmazdı.", "Konser sıkıcıydı, sanatçı hiç iyi değildi."),
    ("Kargo çok hızlı geldi, paketleme kusursuzdu.", "Kargo çok geç geldi, paket parçalanmıştı."),
    ("Öğretmenimiz çok ilham verici ve sabırlı.", "Öğretmenimiz çok sıkıcı ve sabırsız."),
    ("Bu kitabı okurken çok güzel vakit geçirdim.", "Bu kitabı okurken çok sıkıldım, zaman kaybıydı."),
    ("Otel temiz, personel güler yüzlüydü.", "Otel pisti, personel kabaydı."),
    ("Maçı kazandık, inanılmaz bir sevinç yaşadık.", "Maçı kaybettik, büyük bir hayal kırıklığı yaşadık."),
    ("Doktor çok ilgiliydi, kendimi güvende hissettim.", "Doktor çok ilgisizdi, kendimi değersiz hissettim."),
    ("Bu uygulama hayatımı kolaylaştırdı, bayıldım.", "Bu uygulama sürekli çöküyor, nefret ettim."),
    ("Hafta sonu ailemle harika bir piknik yaptık.", "Hafta sonu ailemle berbat bir kavga ettik."),
    ("Sunumum çok başarılı geçti, herkes alkışladı.", "Sunumum rezalet geçti, herkes sıkıldı."),
    ("Bu kahve mükemmel, tam istediğim gibi.", "Bu kahve korkunç, içilecek gibi değil."),
]
olumlu = [a for a, _ in ciftler]
olumsuz = [b for _, b in ciftler]

A_olumlu = aktivasyonlar(olumlu)      # (N, n_katman, d)
A_olumsuz = aktivasyonlar(olumsuz)
yon_vektorleri = (A_olumlu - A_olumsuz).mean(0)          # (n_katman, d) — her katman için bir yön
print("Yön vektörleri:", tuple(yon_vektorleri.shape))

---
## 3) Kavram gerçekten doğrusal mı? Doğrusal sonda (linear probe)

Her katmanın aktivasyonları üzerinde basit bir **lojistik regresyon** eğitip olumlu/olumsuz ayrımını ne kadar iyi yaptığını ölçüyoruz. Basit bir doğrusal sınıflandırıcı yüksek doğruluk alıyorsa, kavram o katmanda **doğrusal olarak ayrılabilir** demektir. Bu da yönlendirme için iyi bir aday katman olduğunu gösterir.

İkinci ölçüt: her örneği yön vektörüne **izdüşürdüğümüzde** (projection) iki grup ne kadar ayrışıyor?

In [ ]:
X_tum = torch.cat([A_olumlu, A_olumsuz]).numpy()          # (2N, n_katman, d)
y_tum = np.array([1] * len(olumlu) + [0] * len(olumsuz))

sonda_dogrulugu, ayrisma = [], []
for l in range(N_KATMAN):
    Xl = X_tum[:, l]
    Xl = (Xl - Xl.mean(0)) / (Xl.std(0) + 1e-6)
    sonda_dogrulugu.append(cross_val_score(LogisticRegression(max_iter=2000, C=0.1), Xl, y_tum, cv=4).mean())

    v = yon_vektorleri[l] / yon_vektorleri[l].norm()
    izdusum_poz = A_olumlu[:, l] @ v
    izdusum_neg = A_olumsuz[:, l] @ v
    birlesik_std = torch.cat([izdusum_poz, izdusum_neg]).std()
    ayrisma.append(((izdusum_poz.mean() - izdusum_neg.mean()) / birlesik_std).item())

fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(sonda_dogrulugu, "o-", color="tab:blue", label="doğrusal sonda doğruluğu (CV)")
ax1.set_ylabel("doğruluk", color="tab:blue"); ax1.set_ylim(0.4, 1.05)
ax2 = ax1.twinx()
ax2.plot(ayrisma, "s--", color="tab:red", label="yön üzerinde ayrışma (Cohen d)")
ax2.set_ylabel("ayrışma", color="tab:red")
ax1.set_xlabel("katman")
ax1.set_title("Duygu kavramı hangi katmanlarda doğrusal olarak kodlanıyor?")
fig.legend(loc="lower right"); plt.show()

Genellikle **ilk katmanlarda** ayrışma zayıftır (model henüz kelime seviyesinde çalışıyor), **orta katmanlarda** zirve yapar, son katmanlarda ise temsil sonraki token tahminine özelleşir. Yönlendirme için genellikle **orta katmanlar** (toplam derinliğin ~%30–60'ı) en iyi sonucu verir.

In [ ]:
YONLENDIRME_KATMANI = int(np.argmax(ayrisma[: int(N_KATMAN * 0.7)]))
print(f"Seçilen katman: {YONLENDIRME_KATMANI} / {N_KATMAN}")

pca = PCA(2).fit(X_tum[:, YONLENDIRME_KATMANI])
nokta = pca.transform(X_tum[:, YONLENDIRME_KATMANI])
plt.figure(figsize=(7, 5))
plt.scatter(nokta[y_tum == 1, 0], nokta[y_tum == 1, 1], c="tab:green", label="olumlu", s=50)
plt.scatter(nokta[y_tum == 0, 0], nokta[y_tum == 0, 1], c="tab:red", label="olumsuz", s=50)
for i in range(len(olumlu)):
    plt.plot([nokta[i, 0], nokta[i + len(olumlu), 0]], [nokta[i, 1], nokta[i + len(olumlu), 1]],
             color="gray", alpha=0.3, lw=0.8)
plt.title(f"Katman {YONLENDIRME_KATMANI} aktivasyonları (PCA)\ngri çizgiler = eşleşmiş çiftler: hepsi benzer yönü gösteriyor")
plt.legend(); plt.show()

---
## 4) Forward hook ile yönlendirme

PyTorch'ta `register_forward_hook`, bir modülün çıktısını **döndürülmeden önce** değiştirmemize izin verir. Seçtiğimiz katmanın çıktısına $\alpha \cdot \hat{\mathbf{v}}$ ekliyoruz:

- $\alpha > 0$ → olumlu yöne it
- $\alpha < 0$ → olumsuz yöne it
- $\alpha = 0$ → orijinal model

Vektörü birim uzunluğa normalize edip residual stream'in tipik normuna göre ölçekliyoruz; böylece α değerleri katmandan katmana karşılaştırılabilir olur.

In [ ]:
class Yonlendirici:
    """Bağlam yöneticisi: with bloğu içinde seçilen katmanın çıktısına α·v eklenir, çıkınca kanca kaldırılır."""

    def __init__(self, katman_no, vektor, alfa):
        self.katman = katmanlar[katman_no]
        self.vektor = vektor
        self.alfa = alfa

    def _kanca(self, modul, girdi, cikti):
        if isinstance(cikti, tuple):                     # bazı transformers sürümleri tuple döndürür
            gizli = cikti[0]
            return (gizli + self.alfa * self.vektor.to(gizli.dtype),) + cikti[1:]
        return cikti + self.alfa * self.vektor.to(cikti.dtype)

    def __enter__(self):
        self.tutamac = self.katman.register_forward_hook(self._kanca)
        return self

    def __exit__(self, *args):
        self.tutamac.remove()


tipik_norm = A_olumlu[:, YONLENDIRME_KATMANI].norm(dim=-1).mean()
duygu_yonu = (yon_vektorleri[YONLENDIRME_KATMANI] / yon_vektorleri[YONLENDIRME_KATMANI].norm() * tipik_norm).to(cihaz)
print(f"Residual stream tipik normu (katman {YONLENDIRME_KATMANI}): {tipik_norm:.1f}")

In [ ]:
@torch.no_grad()
def uret(istem, max_yeni_token=60, sohbet=True):
    if sohbet:
        istem = tokenizer.apply_chat_template([{"role": "user", "content": istem}],
                                              tokenize=False, add_generation_prompt=True)
    girdi = tokenizer(istem, return_tensors="pt").to(cihaz)
    cikti = model.generate(**girdi, max_new_tokens=max_yeni_token, do_sample=False,
                           repetition_penalty=1.2, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(cikti[0, girdi["input_ids"].shape[1]:], skip_special_tokens=True).strip()

istemler = ["Dün gittiğim restoran hakkında kısa bir yorum yaz.",
            "Pazartesi sabahı işe gitmek hakkında ne düşünüyorsun?",
            "Yağmurlu bir günü iki cümleyle anlat."]

for istem in istemler:
    print("═" * 100)
    print(f"İSTEM: {istem}")
    for alfa in [-0.4, 0.0, 0.4]:
        with Yonlendirici(YONLENDIRME_KATMANI, duygu_yonu, alfa):
            cevap = uret(istem)
        etiket = "olumsuz yönde" if alfa < 0 else "olumlu yönde" if alfa > 0 else "orijinal"
        print(f"\n  α = {alfa:+.1f} ({etiket}):\n  {cevap}")

---
## 5) Katsayı (α) taraması ve ölçüm

Yönlendirmenin etkisini gözle değil sayıyla ölçmek için üretilen metinleri bağımsız bir **duygu sınıflandırıcısıyla** puanlıyoruz. Aynı zamanda metnin **akıcılığını** da izlemek gerekir: α çok büyürse model kavramı abartır, sonra da tekrarlayan/anlamsız metin üretmeye başlar. İyi bir yönlendirme bu ikisi arasında denge kurar.

Akıcılık ölçütü olarak, yönlendirilmemiş modelin üretilen metne verdiği **perplexity** değerini kullanıyoruz.

In [ ]:
from transformers import pipeline

duygu_siniflandirici = pipeline("text-classification", model="savasy/bert-base-turkish-sentiment-cased",
                                device=0 if cihaz == "cuda" else -1)

def olumluluk_puani(metin):
    sonuc = duygu_siniflandirici(metin[:512])[0]
    return sonuc["score"] if sonuc["label"].lower().startswith("pos") else 1 - sonuc["score"]

@torch.no_grad()
def perplexity(metin):
    girdi = tokenizer(metin, return_tensors="pt").to(cihaz)
    if girdi["input_ids"].shape[1] < 2:
        return float("nan")
    kayip = model(**girdi, labels=girdi["input_ids"]).loss
    return torch.exp(kayip).item()

satirlar = []
for alfa in [-0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8]:
    for istem in istemler:
        with Yonlendirici(YONLENDIRME_KATMANI, duygu_yonu, alfa):
            cevap = uret(istem, max_yeni_token=50)
        satirlar.append({"α": alfa, "olumluluk": olumluluk_puani(cevap), "perplexity": perplexity(cevap)})

tarama = pd.DataFrame(satirlar).groupby("α").mean()
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(tarama.index, tarama["olumluluk"], "o-", color="tab:green")
ax1.set_ylabel("olumluluk puanı (0 = olumsuz, 1 = olumlu)", color="tab:green"); ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
ax2.plot(tarama.index, tarama["perplexity"], "s--", color="tab:purple")
ax2.set_ylabel("perplexity (yüksek = daha az akıcı)", color="tab:purple")
ax1.set_xlabel("α (yönlendirme katsayısı)")
ax1.set_title("Yönlendirme gücü ↔ akıcılık dengesi")
plt.show()
tarama.round(3)

---
## 6) Başka bir kavram: resmi ↔ samimi üslup

Aynı yöntem herhangi bir kavram için çalışır. Sadece yeni zıt çiftler gerekir.

In [ ]:
uslup_ciftleri = [
    ("Sayın yetkili, talebimin değerlendirilmesini rica ederim.", "Abi şu işi bi hallediver ya."),
    ("Toplantının ertelenmesi hususunda bilginize sunarım.", "Toplantı ertelendi kanka, haberin olsun."),
    ("Göstermiş olduğunuz ilgi için teşekkürlerimi arz ederim.", "Eyvallah ya, çok sağ ol."),
    ("Belirtilen tarihte katılım sağlayamayacağımı bildiririm.", "O gün gelemem ya, kusura bakma."),
    ("Konuyla ilgili görüşlerinizi bekliyorum.", "Ne düşünüyon bu işe, yaz bi."),
    ("Ödemenin en kısa sürede yapılmasını talep ediyorum.", "Parayı bi an önce at artık ya."),
    ("Sunumun ekte yer aldığını bilgilerinize sunarım.", "Sunumu attım bak, bi göz at."),
    ("Yaşanan aksaklık için özür dileriz.", "Pardon ya, bi şey ters gitti."),
    ("Görüşmemizi yarın saat onda gerçekleştirebiliriz.", "Yarın onda takılalım mı?"),
    ("Başvurunuz olumlu olarak değerlendirilmiştir.", "Başvurun kabul edildi, helal!"),
]
A_resmi = aktivasyonlar([a for a, _ in uslup_ciftleri])
A_samimi = aktivasyonlar([b for _, b in uslup_ciftleri])
uslup_vektorleri = (A_resmi - A_samimi).mean(0)

v = uslup_vektorleri[YONLENDIRME_KATMANI]
uslup_yonu = (v / v.norm() * A_resmi[:, YONLENDIRME_KATMANI].norm(dim=-1).mean()).to(cihaz)

cos = F.cosine_similarity(yon_vektorleri[YONLENDIRME_KATMANI], uslup_vektorleri[YONLENDIRME_KATMANI], dim=0)
print(f"Duygu yönü ile üslup yönü arasındaki kosinüs benzerliği: {cos:.3f}  (≈ 0 → birbirinden bağımsız kavramlar)\n")

istem = "Arkadaşına yarınki buluşmayı iptal ettiğini söyleyen kısa bir mesaj yaz."
for alfa, etiket in [(-0.4, "samimi"), (0.0, "orijinal"), (0.4, "resmi")]:
    with Yonlendirici(YONLENDIRME_KATMANI, uslup_yonu, alfa):
        print(f"α = {alfa:+.1f} ({etiket}): {uret(istem)}\n")

### İki yönü aynı anda uygulamak

Yönler (yaklaşık) dik olduğu için toplanabilir: *resmi ama olumsuz*, *samimi ve olumlu* gibi kombinasyonlar.

In [ ]:
istem = "Yeni açılan kafe hakkında bir değerlendirme yaz."
for a_duygu, a_uslup, etiket in [(0.4, 0.4, "olumlu + resmi"), (0.4, -0.4, "olumlu + samimi"),
                                  (-0.4, 0.4, "olumsuz + resmi"), (-0.4, -0.4, "olumsuz + samimi")]:
    with Yonlendirici(YONLENDIRME_KATMANI, duygu_yonu, a_duygu), Yonlendirici(YONLENDIRME_KATMANI, uslup_yonu, a_uslup):
        print(f"[{etiket}]\n{uret(istem)}\n")

---
## 7) Neden önemli? Kullanım alanları ve sınırlar

**Kullanım alanları**
- **Yorumlanabilirlik (interpretability):** Modelin bir kavramı nerede ve nasıl kodladığını anlamak. Sonda + yönlendirme birlikte, "bu yön gerçekten bu kavramı mı temsil ediyor?" sorusuna **nedensel** bir test sağlar.
- **Davranış kontrolü:** Ton, üslup, ayrıntı seviyesi, dil gibi özellikleri ince ayar yapmadan ayarlamak.
- **Güvenlik araştırması:** Dürüstlük, dalkavukluk (sycophancy), halüsinasyon eğilimi gibi davranışlarla ilişkili yönleri bulup izlemek ve azaltmak. Bu konu *Representation Engineering* (Zou vd., 2023) olarak geçer.
- **Sparse Autoencoder (SAE) özellikleri:** Ham fark vektörleri yerine, SAE'lerin bulduğu yorumlanabilir özellik (feature) yönleriyle yönlendirme yapmak (ör. Anthropic'in *Golden Gate Claude* gösterimi).

**Sınırlar**
- Yönler **gürültülüdür**: fark vektörü hedef kavramın yanında çiftler arasındaki başka farkları da taşıyabilir. Daha fazla ve daha iyi eşleşmiş çift bunu azaltır.
- **α hassastır**: küçük α etkisiz, büyük α metni bozar. Her model, katman ve kavram için tarama gerekir.
- Kavram her zaman tek bir doğrusal yönle temsil edilmez; doğrusal temsil hipotezi bir yaklaşımdır.
- Yönlendirme modelin **yapabildiği** davranışları öne çıkarır, olmayan bir yeteneği eklemez.

## Özet

| Adım | Ne yaptık? |
|---|---|
| Aktivasyon okuma | `output_hidden_states=True` ile her katmanın residual stream vektörü |
| Yön çıkarma | Zıt çiftlerin aktivasyon farklarının ortalaması: $\mathbf{v} = \overline{h(+) - h(-)}$ |
| Doğrulama | Doğrusal sonda doğruluğu ve yön üzerindeki ayrışma ile katman seçimi |
| Yönlendirme | `register_forward_hook` ile $h \leftarrow h + \alpha\hat{\mathbf{v}}$ |
| Ölçüm | Bağımsız sınıflandırıcıyla hedef kavram + perplexity ile akıcılık |

**Kaynaklar:** Turner vd. (2023) *Activation Addition: Steering Language Models Without Optimization* · Rimsky vd. (2024) *Steering Llama 2 via Contrastive Activation Addition* · Zou vd. (2023) *Representation Engineering: A Top-Down Approach to AI Transparency* · Templeton vd. (2024) *Scaling Monosemanticity*